# 🗡️ Hey Aragorn Wake Word Training

Train a custom wake word model using micro-wake-word.

**Run each cell in order.**

## Step 0a: Switch to Python 3.10

Colab now defaults to Python 3.12. Several dependencies (piper-phonemize, microWakeWord) require Python 3.10. This cell installs Miniforge with Python 3.10 and **automatically restarts the runtime**.

**After the auto-restart, continue from Step 0b.**

In [ ]:
# Installs Miniforge3-23.3.1 which ships with Python 3.10.
# condacolab triggers an automatic runtime restart — no manual action needed.
!pip install -q condacolab
import condacolab
condacolab.install_from_url(
    "https://github.com/conda-forge/miniforge/releases/download/23.3.1-1/Miniforge3-23.3.1-1-Linux-x86_64.sh"
)


## ↩️ Auto-Restart Happened

The runtime was automatically restarted with Python 3.10. Continue from **Step 0b**.

## Step 0b: Disk Cleanup + Install numpy/scipy

Frees ~2–3 GB of unused system files, then installs numpy and scipy via conda **before** any pip activity to avoid the mid-cell restart prompt.

In [ ]:
import sys, shutil, subprocess

# Verify Python 3.10
assert sys.version_info[:2] == (3, 10), f"Expected Python 3.10, got {sys.version}. Re-run Step 0a."
print(f"Python {sys.version_info.major}.{sys.version_info.minor}: OK")

# ── Disk cleanup ───────────────────────────────────────────────────────────
print("\nFreeing disk space...")
for path in ['/usr/share/doc', '/usr/share/man', '/usr/share/locale',
             '/usr/lib/google-cloud-sdk', '/usr/local/android-sdk',
             '/usr/local/julia-1.9.4', '/content/sample_data']:
    shutil.rmtree(path, ignore_errors=True)
!apt-get clean -qq
!apt-get autoremove -y -qq
!pip cache purge -q 2>/dev/null
print("Disk cleanup done")
!df -h / | tail -1

# ── Install numpy + scipy via conda FIRST ─────────────────────────────────
# Installing via conda before any pip activity means pip won't need to
# reinstall numpy later, eliminating the mid-cell restart prompt entirely.
print("\nInstalling numpy + scipy via conda...")
subprocess.run(['conda', 'install', '-y', '-q',
                'numpy=1.26.4', 'scipy=1.13.1', '-c', 'conda-forge'], check=True)

import numpy as np, scipy
print(f"numpy {np.__version__}: OK")
print(f"scipy {scipy.__version__}: OK")
print("\n\u2705 Ready for Step 1!")


## Step 1: Check GPU

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Step 2: Install All Packages

No mid-cell restart prompt this time — numpy/scipy were already installed via conda in Step 0b so pip won't reinstall them.

After this cell completes, **restart the runtime** (Runtime → Restart session) so TensorFlow loads cleanly, then continue from Step 3.

In [ ]:
# ── System deps ────────────────────────────────────────────────────────────
print("Installing system dependencies...")
!apt-get install -y -q espeak-ng libespeak-ng-dev

# ── piper-tts: explicit install with visible output ────────────────────────
# Must be installed before TF/etc. so its deps resolve correctly on Python 3.10
print("\nInstalling piper-tts (visible output so failures are obvious)...")
!pip install piper-tts

print("\nVerifying piper import...")
!python -c "from piper import PiperVoice, SynthesisConfig; print('piper: OK')"

# ── Remove conflicting Colab pre-installs ──────────────────────────────────
print("\nRemoving conflicting pre-installed packages...")
!pip uninstall -y jax jaxlib tensorstore tensorflow-decision-forests tensorflow-text opencv-python opencv-python-headless opencv-contrib-python shap ydf grain pytensor xarray-einstats rasterio tobler cupy-cuda12x 2>/dev/null

# ── TensorFlow + Keras 2 compat layer ─────────────────────────────────────
# tf-keras must be KEPT — it's the Keras 2 compatibility layer.
# Without it TF 2.16 defaults to Keras 3 which breaks model.evaluate() in microWakeWord.
print("\nInstalling TensorFlow...")
!pip install --quiet tensorflow==2.16.2 protobuf==4.25.3 ml-dtypes==0.3.2 tf-keras 2>/dev/null

# ── Remaining dependencies ─────────────────────────────────────────────────
!pip install --quiet onnxruntime 2>/dev/null
!pip install --quiet pyyaml datasets mmap-ninja tqdm audiomentations webrtcvad-wheels huggingface_hub 2>/dev/null

print("\n\u2705 All packages installed!")
print("\n\u26a0\ufe0f Restart the runtime now: Runtime \u2192 Restart session")
print("   Then continue from Step 3.")


## ⚠️ Restart Required

**Restart the runtime now!**

Runtime → Restart session (Ctrl+M .)

Then continue from **Step 3**.

## Step 3: Verify Install & Clone Repositories

In [ ]:
import subprocess, os, sys

assert sys.version_info[:2] == (3, 10), f"Wrong Python: {sys.version}. Did you run Step 0a?"
print(f"Python {sys.version_info.major}.{sys.version_info.minor}: OK")

import numpy as np
assert np.__version__ == '1.26.4', f"Bad numpy: {np.__version__}"
print(f"numpy {np.__version__}: OK")

import scipy
from scipy.signal import resample
from scipy.io import wavfile
print(f"scipy {scipy.__version__}: OK")

import tensorflow as tf
print(f"tensorflow {tf.__version__}: OK")

r = subprocess.run([sys.executable, '-c',
    'from piper import PiperVoice, SynthesisConfig; print("piper: OK")'],
    capture_output=True, text=True)
print(r.stdout.strip() if r.returncode == 0 else f"piper FAILED: {r.stderr}")

# Clone microWakeWord
if not os.path.exists('microWakeWord'):
    subprocess.run(['git', 'clone', 'https://github.com/kahrendt/microWakeWord.git'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', 'microWakeWord'], check=True)
print("microWakeWord: OK")

# Clone piper-sample-generator (just clone — piper-tts already installed)
if not os.path.exists('piper-sample-generator'):
    subprocess.run(['git', 'clone', 'https://github.com/rhasspy/piper-sample-generator.git'], check=True)
print("piper-sample-generator: OK")

print("\n\u2705 All set!")


## Step 4: Download Piper Voice Model

In [ ]:
import os, urllib.request

os.makedirs('piper-sample-generator/models', exist_ok=True)
url  = 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
path = 'piper-sample-generator/models/en_US-libritts_r-medium.pt'

if not os.path.exists(path):
    print('Downloading...')
    urllib.request.urlretrieve(url, path)
    print('\u2705 Done!')
else:
    print('\u2705 Already exists')


## Step 5: Configure

In [ ]:
TARGET_WORD    = 'hey_air_uh_gorn'
NUM_SAMPLES    = 1000
TRAINING_STEPS = 10000

print(f'Word:    {TARGET_WORD}')
print(f'Samples: {NUM_SAMPLES}')
print(f'Steps:   {TRAINING_STEPS}')


## Step 6: Generate Test Sample

Listen and confirm pronunciation. Adjust `TARGET_WORD` in Step 5 if needed.

Tips: underscores between syllables (`hey_air_uh_gorn`), `sh`/`ch`/`th` for those sounds, `ee`/`oo` for long vowels.

In [ ]:
import subprocess, os, sys
from IPython.display import Audio, display

os.makedirs('generated_samples', exist_ok=True)
print('Generating 1 test sample...')

env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', '1', '--batch-size', '1',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    wavs = sorted([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    if wavs:
        print(f'Phonetic spelling: "{TARGET_WORD}"')
        print('Adjust TARGET_WORD in Step 5 if this does not sound right.\n')
        display(Audio(os.path.join('generated_samples', wavs[0])))
    else:
        print('No .wav files found.')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 7: Generate All Samples

In [ ]:
import subprocess, os, sys

print(f'Generating {NUM_SAMPLES} samples...')
env = {**os.environ, 'PYTHONPATH': os.path.abspath('piper-sample-generator')}
cmd = [sys.executable, '-m', 'piper_sample_generator',
       TARGET_WORD, '--max-samples', str(NUM_SAMPLES), '--batch-size', '100',
       '--model', 'piper-sample-generator/models/en_US-libritts_r-medium.pt',
       '--output-dir', 'generated_samples']
result = subprocess.run(cmd, capture_output=True, text=True, env=env)

if result.returncode == 0:
    count = len([f for f in os.listdir('generated_samples') if f.endswith('.wav')])
    print(f'\u2705 Generated {count} samples!')
else:
    print('STDOUT:', result.stdout)
    print('STDERR:', result.stderr)


## Step 8: Download Augmentation Data

In [ ]:
import os

os.makedirs('mit_rirs', exist_ok=True)
if not os.listdir('mit_rirs'):
    print('Downloading RIRs + pointsource noises (~1.3 GB)...')
    !wget --progress=bar:force -O /tmp/rirs_noises.zip https://www.openslr.org/resources/28/rirs_noises.zip
    !unzip -q /tmp/rirs_noises.zip -d mit_rirs
    print('Extracted!')
else:
    print('Already downloaded')

noise_dir = 'mit_rirs/RIRS_NOISES/pointsource_noises'
n = len([f for f in os.listdir(noise_dir) if f.endswith('.wav')]) if os.path.exists(noise_dir) else 0
print(f'\u2705 {n} background noise files ready' if n else '\u274c pointsource_noises missing')


## 🧹 Disk Cleanup (After Step 8)

Deletes the zip archive (already extracted) and caches. Run before Step 9.

In [ ]:
import os, subprocess, sys
if os.path.exists('/tmp/rirs_noises.zip'):
    os.remove('/tmp/rirs_noises.zip')
    print('Deleted /tmp/rirs_noises.zip')
subprocess.run([sys.executable, '-m', 'pip', 'cache', 'purge'], capture_output=True)
print('pip cache purged')
!df -h / | tail -1


## Step 9: Generate Spectrograms

In [ ]:
import os, sys
if 'microWakeWord' not in sys.path:
    sys.path.insert(0, 'microWakeWord')

from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap

print('Generating spectrograms...')

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=False,
    random_split_seed=10,
    split_count=0.1,
)

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.1,
        'TanhDistortion': 0.1,
        'PitchShift': 0.1,
        'BandStopFilter': 0.1,
        'AddColorNoise': 0.1,
        'AddBackgroundNoise': 0.75,
        'Gain': 1.0,
        'RIR': 0.5,
    },
    impulse_paths=['mit_rirs'],
    background_paths=['mit_rirs/RIRS_NOISES/pointsource_noises'],
    background_min_snr_db=-5,
    background_max_snr_db=10,
    min_jitter_s=0.195,
    max_jitter_s=0.205,
)

os.makedirs('generated_augmented_features/training', exist_ok=True)

spectrograms = SpectrogramGeneration(
    clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10,
)

RaggedMmap.from_generator(
    out_dir='generated_augmented_features/training/wakeword_mmap',
    sample_generator=spectrograms.spectrogram_generator(split='train', repeat=2),
    batch_size=100,
    verbose=True,
)
print('\u2705 Done!')


## Step 10: Download Negative Datasets

In [ ]:
import os, zipfile
from huggingface_hub import hf_hub_download, list_repo_files

print('Discovering negative datasets on HuggingFace...')
os.makedirs('negative_datasets', exist_ok=True)

repo_id, repo_type = 'kahrendt/microwakeword', 'dataset'
zip_files = [f for f in list_repo_files(repo_id, repo_type=repo_type) if f.endswith('.zip')]
print(f'Found: {zip_files}')

for fname in zip_files:
    base    = os.path.splitext(os.path.basename(fname))[0]
    out_dir = f'negative_datasets/{base}'
    if not os.path.exists(out_dir):
        print(f'Downloading {fname}...')
        local = hf_hub_download(repo_id=repo_id, filename=fname, repo_type=repo_type)
        with zipfile.ZipFile(local, 'r') as zf:
            zf.extractall('negative_datasets')
        print(f'  done: {base}')
    else:
        print(f'  exists: {base}')

neg_dirs = [d for d in os.listdir('negative_datasets') if os.path.isdir(f'negative_datasets/{d}')]
print(f'\n\u2705 Directories: {neg_dirs}')


## 🧹 Disk Cleanup (After Step 10)

Removes HuggingFace's local zip cache — datasets are already extracted.

In [ ]:
import os, shutil

hf_cache = os.path.expanduser('~/.cache/huggingface')
if os.path.exists(hf_cache):
    size = sum(os.path.getsize(os.path.join(dp,f)) for dp,_,fs in os.walk(hf_cache) for f in fs)
    shutil.rmtree(hf_cache)
    print(f'Deleted HuggingFace cache ({size/1e9:.1f} GB freed)')
else:
    print('No HuggingFace cache found')
!df -h / | tail -1


## Step 11: Create Config

In [ ]:
import yaml, os

# Filter out __MACOSX and other macOS metadata folders from zip extraction
neg_dirs   = sorted([d for d in os.listdir('negative_datasets')
                     if os.path.isdir(f'negative_datasets/{d}') and not d.startswith('__')])
eval_dirs  = [d for d in neg_dirs if 'eval' in d]
train_dirs = [d for d in neg_dirs if 'eval' not in d]
print(f'Train negatives : {train_dirs}')
print(f'Eval  negatives : {eval_dirs}')

neg_features = []
for d in train_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}', 'sampling_weight': 10.0,
                         'penalty_weight': 1.0, 'truth': False,
                         'truncation_strategy': 'random', 'type': 'mmap'})
for d in eval_dirs:
    neg_features.append({'features_dir': f'negative_datasets/{d}', 'sampling_weight': 0.0,
                         'penalty_weight': 1.0, 'truth': False,
                         'truncation_strategy': 'split', 'type': 'mmap'})

pos_feature = {'features_dir': 'generated_augmented_features/training',
               'sampling_weight': 2.0, 'penalty_weight': 1.0, 'truth': True,
               'truncation_strategy': 'truncate_start', 'type': 'mmap'}

config = {
    'window_step_ms': 10,
    'train_dir': 'trained_models/wakeword',
    'spectrogram_length': 204,
    'stride': 3,
    'features': [pos_feature] + neg_features,
    'training_steps': [TRAINING_STEPS],
    'positive_class_weight': [1],
    'negative_class_weight': [20],
    'learning_rates': [0.001],
    'batch_size': 128,
    'eval_step_interval': 500,
    'clip_duration_ms': 1500,
    'target_minimization': 0.9,
    'minimization_metric': '',
    'maximization_metric': 'average_viable_recall',
}

with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)
print(f'\u2705 Config saved with {len(config["features"])} feature sources!')


## Step 12: Train Model

This takes 1-3 hours. Go get coffee!

In [ ]:
import subprocess, sys, os

print('Starting training...')
print(f'~{TRAINING_STEPS // 10000} hour(s)...')

train_env = {
    **os.environ,
    'TF_USE_LEGACY_KERAS': '1',
    'TF_FORCE_GPU_ALLOW_GROWTH': 'true',
    'TF_CPP_MIN_LOG_LEVEL': '2',
}

train_cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config=training_parameters.yaml',
    '--train=1',
    '--restore_checkpoint', '0',
    '--test_tf_nonstreaming', '0',
    '--test_tflite_nonstreaming', '0',
    '--test_tflite_nonstreaming_quantized', '0',
    '--test_tflite_streaming', '0',
    '--test_tflite_streaming_quantized', '1',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]

result = subprocess.run(train_cmd, stderr=subprocess.PIPE, text=True, env=train_env)

if result.returncode == 0:
    print('\u2705 Training complete!')
else:
    print('\u274c Training failed. Error output:')
    print(result.stderr[-3000:] if len(result.stderr) > 3000 else result.stderr)


## Step 13: Download Model

In [ ]:
import os
from google.colab import files

model_path = 'trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'

if os.path.exists(model_path):
    print(f'\u2705 Model: {os.path.getsize(model_path)/1024:.1f} KB')
    files.download(model_path)
    print('\n\U0001f389 Done! Check your downloads.')
else:
    print('\u274c Model not found - check training output above')
